# What drives the price of a car?

![](images/kurt.jpeg)

**OVERVIEW**

In this application, you will explore a dataset from Kaggle. The original dataset contained information on 3 million used cars. The provided dataset contains information on 426K cars to ensure speed of processing.  Your goal is to understand what factors make a car more or less expensive.  As a result of your analysis, you should provide clear recommendations to your client -- a used car dealership -- as to what consumers value in a used car.

### CRISP-DM Framework

<center>
    <img src = images/crisp.png width = 50%/>
</center>


To frame the task, throughout our practical applications, we will refer back to a standard process in industry for data projects called CRISP-DM.  This process provides a framework for working through a data problem.  Your first step in this application will be to read through a brief overview of CRISP-DM [here](https://mo-pcco.s3.us-east-1.amazonaws.com/BH-PCMLAI/module_11/readings_starter.zip).  After reading the overview, answer the questions below.

### Business Understanding

From a business perspective, we are tasked with identifying key drivers for used car prices.  In the CRISP-DM overview, we are asked to convert this business framing to a data problem definition.  Using a few sentences, reframe the task as a data task with the appropriate technical vocabulary.

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt


In [3]:
vehicles = pd.read_csv('./data/vehicles.csv')
vehicles.head()


,id,region,price,year,manufacturer,model,condition,cylinders,fuel,odometer,title_status,transmission,VIN,drive,size,type,paint_color,state
0,7222695916,prescott,6000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,az
1,7218891961,fayetteville,11900,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ar
2,7221797935,florida keys,21000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,fl
3,7222270760,worcester / central MA,1500,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ma
4,7210384030,greensboro,4900,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nc


**Business Goal :** AI/ML model that determine the features that drive the price of the used cars from the provided vehicles data set of about 426k rows and 18 features/columns. The target variable for the model will be the price of the car and some the input/independent variables year, model, condition, manufacturer, cylinders, fuel, odometer, title_status, condition. This can be classified as a supervised learning problem and the target variable price is continuous numeric quantity. Task is to build an AI/ML model to predict the price of the used cars using the different regression techniques and intepret the significant features and their coefficients as the output is to rank the features in order of significance.

### Data Understanding

After considering the business understanding, we want to get familiar with our data.  Write down some steps that you would take to get to know the dataset and identify any quality issues within.  Take time to get to know the dataset and explore what information it contains and how this could be used to inform your business understanding.

In [4]:
print(vehicles.shape)
print(vehicles.dtypes)

vehicles.info()

vehicles.describe()

# The data set has 426880 rows and 18 columns. The column price is numeric and is the target or the dependant variable.
# Id and VIN are identifier columns
# year and odometer are numeric columns/inputs/independent variables
# rest 13 columns/inputs/independent varaibles are of object data type.

(426880, 18)
id                int64
region           object
price             int64
year            float64
manufacturer     object
model            object
condition        object
cylinders        object
fuel             object
odometer        float64
title_status     object
transmission     object
VIN              object
drive            object
size             object
type             object
paint_color      object
state            object
dtype: object
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 426880 entries, 0 to 426879
Data columns (total 18 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   id            426880 non-null  int64  
 1   region        426880 non-null  object 
 2   price         426880 non-null  int64  
 3   year          425675 non-null  float64
 4   manufacturer  409234 non-null  object 
 5   model         421603 non-null  object 
 6   condition     252776 non-null  object 
 7   cylinders     249202 non-null

,id,price,year,odometer
count,4.268800e+05,4.268800e+05,425675.000000,4.224800e+05
mean,7.311487e+09,7.519903e+04,2011.235191,9.804333e+04
std,4.473170e+06,1.218228e+07,9.452120,2.138815e+05
min,7.207408e+09,0.000000e+00,1900.000000,0.000000e+00
25%,7.308143e+09,5.900000e+03,2008.000000,3.770400e+04
50%,7.312621e+09,1.395000e+04,2013.000000,8.554800e+04
75%,7.315254e+09,2.648575e+04,2017.000000,1.335425e+05
max,7.317101e+09,3.736929e+09,2022.000000,1.000000e+07


In [5]:
# Check for null data
vehicles.isnull().sum()

#Calculate the percentages of null values for each column
(vehicles.isnull().mean() * 100).round(2)

# Columns having more than 20% null values

for col in vehicles.columns:
    if (vehicles[col].isnull().mean() * 100) > 20:
        print(col + ":" + str((vehicles[col].isnull().mean() * 100).round(2)))




condition:40.79
cylinders:41.62
VIN:37.73
drive:30.59
size:71.77
type:21.75
paint_color:30.5


# The null values for the columns condition, cylinders , VIN , drive, size , type and paint_color is more than 30% of the total values in the respective columns.



In [6]:
import matplotlib; matplotlib.use('Agg')


# Check the min ,max , mean , median for the target column price.

print(vehicles['price'].describe())

modelUniqueValues = vehicles['model'].unique()
print('Unique values in model column : ' + str(len(modelUniqueValues)))

# Check the count of price value = 0
zeroPriceCount = vehicles[vehicles['price'] == 0]['price'].count()
print('Records with 0$ as the price : ' + str(zeroPriceCount))

# dataset with rows having same VIN, potential duplicates
VINByCount = vehicles['VIN'].value_counts()
print(VINByCount[VINByCount > 1].count())

OdometerByCounteq0orge300k = vehicles[(vehicles['odometer'] == 0) | (vehicles['odometer'] >= 10000000)].groupby('odometer')['odometer'].count()
print(OdometerByCounteq0orge300k)


# Creating age column as per current year
vehicles['age'] = 2026-vehicles['year']
df = vehicles.copy()
df.shape

# Dataframe to get the vehicles for price >= 500 and less than 100k, odometer reading > 0 and < 300k, year > 1990

v = df[(df['price'] >= 500) & (df['price'] <= 100000) & (df['odometer'] > 0) & (df['odometer'] < 300000) & (df['year'] >= 1990)].copy()
plt.rcParams.update({'figure.dpi':100,'axes.grid':True,'grid.alpha':0.3,'axes.spines.top':'False','axes.spines.right':False})

#1 Missing Values chart
miss = (df.isna().mean()*100).sort_values()
fig,ax = plt.subplots(figsize=(8,6))
C='#2b6cb0'

ax.barh(miss.index, miss.values, color=C)
ax.set_title('Missing values by column(%)')
ax.set_xlabel('% missing')
plt.tight_layout()
plt.savefig('./images/missingvalues.png')
plt.show()
plt.close()

#2 Price distribution Raw vs filtered
fig,ax = plt.subplots(1,2,figsize=(12,6))
ax[0].hist(vehicles['price'].clip(upper=vehicles['price'].quantile(.999)),bins=60,color='#c05621')
ax[0].set_title('Raw Price Distribution')
ax[0].set_xlabel('Price')
ax[0].set_ylabel('Count')
ax[1].hist(v['price'],bins=60,color=C)
ax[1].set_title('Filtered Price Distribution')
ax[1].set_xlabel('Price')
ax[1].set_ylabel('Count')
plt.tight_layout()
plt.savefig('./images/pricedistribution.png')
plt.show()
plt.close()






count    4.268800e+05
mean     7.519903e+04
std      1.218228e+07
min      0.000000e+00
25%      5.900000e+03
50%      1.395000e+04
75%      2.648575e+04
max      3.736929e+09
Name: price, dtype: float64
Unique values in model column : 29650
Records with 0$ as the price : 32895
40280
odometer
0.0           1965
10000000.0      50
Name: odometer, dtype: int64


# The null values for the columns condition, cylinders , VIN , drive, size , type and paint_color is more than 30% of the total values in the respective columns.
# The column id is for identification and can be dropped from the dataset as this feature does not impact the price of the used car.
# Column model has 29649 categories which is very high cardinality for any one hot encoder.
# VIN number column has a lot of null values and will not add much to the car price and can be dropped from the dataset.
# Size column has about 71.77% of null values and will impact the ability of the model to predict the price efficiently. The column can be dropped from the data set.
# The columns/features (condition, cylinders, drive,type) that are important and has 20% more than null values need to marked as unknown or any categorical value in the dataset.
# Other columns which have less than 5% of data missing can be handled appropriately.
# Histogram plot for the distribution of target price taking all the rows is more skewed to the left and also have outliers. PLot with price values filtered between 500$ to 100k$ is more stable.
# There are 32895 rows with price value equals to 0.
# Duplicate VIN number records counts is 40280.
#Odometer column has values equal to 0 and some values equal to 10,000,000 miles which does not look right.


In [7]:
#3 plots of Price, Odometer, age
fig,ax = plt.subplots(1,2,figsize=(15,6))
sample=v.sample(20000,random_state=1)
ax[0].scatter(sample['odometer'],sample['price'],color='#c05621')
ax[0].set_title('Price vs Odometer')
ax[0].set_xlabel('Odometer')
ax[0].set_ylabel('Price')

ax[1].scatter(sample['age'],sample['price'],color=C)
ax[1].set_title('Price vs Age')
ax[1].set_xlabel('Age')
ax[1].set_ylabel('Price')
plt.tight_layout()
plt.savefig('./images/priceageodometer.png')
plt.show()
plt.close()

In [8]:
#Median prices by categories
fig,ax = plt.subplots(2,2,figsize=(24,12))
columns = ['condition','type','drive','fuel']

mc = v.groupby(columns[0])['price'].median().sort_values()
mt = v.groupby(columns[1])['price'].median().sort_values()
md = v.groupby(columns[2])['price'].median().sort_values()
mf = v.groupby(columns[3])['price'].median().sort_values()

ax[0,0].barh(mc.index, mc.values, color=C)
ax[0,0].set_title('Median Price by Condition')
ax[0,0].set_xlabel('Median Price')
ax[0,0].set_ylabel('Group by Condition')

ax[1,0].barh(mt.index,mt.values,color='#c05621')
ax[1,0].set_title('Median Price by Type')
ax[1,0].set_xlabel('Median Price')
ax[1,0].set_ylabel('Group by Type')

ax[0,1].barh(md.index,md.values,color=C)
ax[0,1].set_title('Median Price by Drive')
ax[0,1].set_xlabel('Median Price')
ax[0,1].set_ylabel('Group by Drive')

ax[1,1].barh(mf.index,mf.values,color='#c05621')
ax[1,1].set_title('Median Price by Fuel')
ax[1,1].set_xlabel('Median Price')
ax[1,1].set_ylabel('Group by Fuel')

plt.tight_layout()
plt.savefig('./images/medianPlots.png')
plt.show()
plt.close()



In [9]:
# Correlation of numerical columns
numcols = ['price','odometer','age']
corr = v[numcols].corr()
fig,ax = plt.subplots(figsize=(8,4))

im = ax.imshow(corr,cmap='RdBu_r',vmin=-1,vmax=1)
ax.set_xticks(np.arange(len(corr.columns)))
ax.set_yticks(np.arange(len(corr.columns)))
ax.set_xticklabels(corr.columns)
ax.set_yticklabels(corr.columns)

for i in range(len(corr.columns)):
    for j in range(len(corr.columns)):
        ax.text(j,i,round(corr.iloc[i,j],2),ha='center',va='center',color='w')
ax.set_title('Correlation of Numerical Columns')
plt.colorbar(im,fraction=.046)

# Correlation of price and age
CorrpriceAge = round(v[['price','age']].corr().iloc[0,1],3)
print('Correlation of price and age : ' + str(CorrpriceAge))

# Correlation of price and odometer
CorrpriceOdometer = round(v[['price','odometer']].corr().iloc[0,1],3)
print('Correlation of price and odometer : ' + str(CorrpriceOdometer))

plt.tight_layout()
plt.savefig('./images/correlation.png')
plt.show()
plt.close()


Correlation of price and age : -0.585
Correlation of price and odometer : -0.549


# Group by condition median plot shows us that the price is better for newer condition cars.
# Group by Type median shows that the if the type is pickup and truck then the median prices are more when compared to other types.
# Group by drive median plot indicates higher prices for 4wd than the other fwd or rwd cars.
#Group by fuel type median plot indicates highest price for diesel cars followed by electric, gas and hybrid.
# Scatter plots of price and odometer , price and age shows a steep decline as the number of years and mileage increases. the heap map also confirms the negative relationship between price and age , price and odometer.

### Data Preparation

After our initial exploration and fine-tuning of the business understanding, it is time to construct our final dataset prior to modeling.  Here, we want to make sure to handle any integrity issues and cleaning, the engineering of new features, any transformations that we believe should happen (scaling, logarithms, normalization, etc.), and general preparation for modeling with `sklearn`.

In [10]:
# Data Preparation for modeling
df = pd.read_csv('./data/vehicles.csv')
originalLengthDF = len(df)
df.shape

(426880, 18)

In [11]:
# Remove the columns (id,VIN,size,model)
# VIN,ID : considered as identifier columns
# size : lot of empty values
# model:high cardinality for mapping

dropcols = ['id','VIN','size','model']
df.drop(columns=dropcols,inplace=True)
df.shape


(426880, 14)

In [12]:
# Remove the 0$ and high dollar amount proce values.
# keep the rows between 1000$ to 100k$

df = df[(df['price'] >= 1000) & (df['price'] <= 100000)]
print("Number of rows after retaining records between $1000 and $100k : " + str(df.shape))

PercentageRowsRetained = round((len(df)/originalLengthDF)*100,0)
print('Percentage of rows Retained : ' + str(PercentageRowsRetained))


Number of rows after retaining records between $1000 and $100k : (379910, 14)
Percentage of rows Retained : 89.0


In [13]:
# Keep rows between year 1990 to current year 2026
df = df[df['year'] >= 1990]
print("Number of rows retained after removing records with year < 1990 " + str(df.shape))
#Keep Odometer reading greater than 0 and less than equal to 300k
df = df[(df['odometer'] > 0) & (df['odometer'] <= 300000)]
print("Number of rows retained after removing the records with odometer reading of 0 and greater than 300k " + str(df.shape))
# Create a column age of vehicle by subtracting from current year.
df['age'] = 2026-df['year']
# remove the year column as is repalced by age
df.drop(columns=['year'],inplace=True)
df.shape


Number of rows retained after removing records with year < 1990 (367097, 14)
Number of rows retained after removing the records with odometer reading of 0 and greater than 300k (362073, 14)


(362073, 14)

In [14]:
#Handling mising categoricals. Cyliners/conditions/drive/pain_color has about 30 to 40% missing values. Removing all of them will
#result in many missing values. Assign a value 'unknown' to all the nulls in the categorical columns.

catCols = ['condition','cylinders','drive','fuel','paint_color','type','manufacturer',
           'title_status','transmission','state','region']
for col in catCols:
    df[col] = df[col].fillna('unknown').astype(str).str.strip().str.lower()

# Reduce the High cardinality region from 404 to top 30 as per frequency and bucket the rest as other

top_regions =  df['region'].value_counts().nlargest(30).index
df['region'] = np.where(df['region'].isin(top_regions), df['region'], 'other')

print(f'Made the regions to top 30 plus "other" :  '
      f'{df["region"].nunique()} levels')






Made the regions to top 30 plus "other" :  31 levels


In [15]:
# Drop the residuals missing values in low missing columns
beforeCount = len(df)
print(f"Count before removing residuals : {len(df)}")

df = df.dropna()
print(f"Count after removing residuals : {len(df)}")


Count before removing residuals : 362073
Count after removing residuals : 362073


In [16]:
print("The model ready dataframe has the following columns : ")
print(df.columns)

print("Shape of the model ready dataframe : ")
print(df.shape)

print(f"Number of null values : {df.isna().sum().sum()}")

print("\nprice : " , df['price'].describe()[['mean','50%','min','max']])
print("\nodometer : " , df['odometer'].describe()[['mean','50%','min','max']])
print("\nage : " , df['age'].describe()[['mean','50%','min','max']])

df.to_csv('./data/df_clean.csv',index=False)
print(f"Saved clean data as  : df_clean.csv " )


The model ready dataframe has the following columns : 
Index(['region', 'price', 'manufacturer', 'condition', 'cylinders', 'fuel',
       'odometer', 'title_status', 'transmission', 'drive', 'type',
       'paint_color', 'state', 'age'],
      dtype='object')
Shape of the model ready dataframe : 
(362073, 14)
Number of null values : 0

price :  mean     19451.146926
50%      15995.000000
min       1000.000000
max     100000.000000
Name: price, dtype: float64

odometer :  mean     92959.4881
50%      88617.0000
min          1.0000
max     300000.0000
Name: odometer, dtype: float64

age :  mean    13.687223
50%     13.000000
min      4.000000
max     36.000000
Name: age, dtype: float64
Saved clean data as  : df_clean.csv 


### Modeling

With your (almost?) final dataset in hand, it is now time to build some models.  Here, you should build a number of different regression models with the price as the target.  In building your models, you should explore different parameters and be sure to cross-validate your findings.

In [17]:
# Use the clean data and convert the categorical columns to numerical forms and also split the data into train and test to avoid leakage

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
import joblib

vehicles_clean  = pd.read_csv('./data/df_clean.csv')
print(vehicles_clean.shape)
print(vehicles_clean.columns)


# Train/Test data to prevent leakage

X = vehicles_clean.drop(columns=['price'])
y = vehicles_clean['price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

# Use Standard scaler for numerical data and OnehotEncoder for categorical data

num_cols = ['odometer','age']
cat_cols = ['condition','cylinders','drive','fuel','paint_color','type','manufacturer',
           'title_status','transmission','state','region']

pre_process = ColumnTransformer([
    ('num', Pipeline([('scale' , StandardScaler()),
                      ('poly',PolynomialFeatures(2,include_bias=False))]), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore',min_frequency = 50 , sparse_output = True), cat_cols)
])

#FIT on train only but transform both Train and Test data

Xtrain_encoded = pre_process.fit_transform(X_train)
Xtest_encoded = pre_process.transform(X_test)

feature_names = pre_process.get_feature_names_out()

#Using Joblib python library to store the encoded values of pre process stage.
#This will help in reducing the time for processing when modelling.

joblib.dump((Xtrain_encoded,Xtest_encoded,y_train.values,y_test.values,pre_process.get_feature_names_out()),'./data/prep.joblib')

Xtrain_encoded_df = pd.DataFrame(Xtrain_encoded.toarray(), columns=feature_names)

print(f"Encoded train matrix shape {Xtrain_encoded_df.shape}")
print(f"Encoded test matrix shape {Xtest_encoded.shape}")

Xtrain_encoded_df.columns



(362073, 14)
Index(['region', 'price', 'manufacturer', 'condition', 'cylinders', 'fuel',
       'odometer', 'title_status', 'transmission', 'drive', 'type',
       'paint_color', 'state', 'age'],
      dtype='object')
(289658, 13)
(72415, 13)
(289658,)
(72415,)
Encoded train matrix shape (289658, 190)
Encoded test matrix shape (72415, 190)


Index(['num__odometer', 'num__age', 'num__odometer^2', 'num__odometer age',
       'num__age^2', 'cat__condition_excellent', 'cat__condition_fair',
       'cat__condition_good', 'cat__condition_like new', 'cat__condition_new',
       ...
       'cat__region_pittsburgh', 'cat__region_reno / tahoe',
       'cat__region_rochester', 'cat__region_sarasota-bradenton',
       'cat__region_south florida', 'cat__region_south jersey',
       'cat__region_st louis, mo', 'cat__region_tampa bay area',
       'cat__region_tucson', 'cat__region_washington, dc'],
      dtype='object', length=190)

In [18]:
import joblib
from sklearn.linear_model import RidgeCV
from sklearn.linear_model import LassoCV
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import r2_score
from sklearn.pipeline import Pipeline

rows=[]
fitted={}

# Modelling the data using RidgeCV (L2) Regression

X_train,X_test,y_train,y_test,feature_names = joblib.load('./data/prep.joblib')
ridge_m = RidgeCV(alphas=np.logspace(-1,4,15)).fit(X_train,y_train)
y_ridge_pred = ridge_m.predict(X_test)


print(f"Ridge alpha = {ridge_m.alpha_:.2f} , R2 = {r2_score(y_test,y_ridge_pred):.3f} "
        f"RMSE = {np.sqrt(mean_squared_error(y_test,y_ridge_pred)):.0f} "
        f"MAE = {mean_absolute_error(y_test,y_ridge_pred) : .0f} "
        f"non zero feat : {int((np.abs(ridge_m.coef_) > 1e-6).sum())} ")

cv_rmse = np.sqrt(-cross_val_score(ridge_m, X_train, y_train, scoring='neg_mean_squared_error', cv=3)).mean()
rmse = np.sqrt(mean_squared_error(y_test,y_ridge_pred))
mae=mean_absolute_error(y_test, y_ridge_pred)
r2=r2_score(y_test, y_ridge_pred)
nonzero_ridge = int((np.abs(ridge_m.coef_) > 1e-6).sum())
alpha_ridge = getattr(ridge_m,'alpha_',np.nan)

joblib.dump((ridge_m.coef_,feature_names),'./data/ridge_coef.joblib')

rows.append(['Ridge (L2)', cv_rmse, rmse,mae,r2,nonzero_ridge,alpha_ridge])
fitted['Ridge (L2)'] = ridge_m

Ridge alpha = 2.68 , R2 = 0.735 RMSE = 7370 MAE =  5119 non zero feat : 190 


In [19]:
import joblib
from sklearn.linear_model import RidgeCV
from sklearn.linear_model import LassoCV
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import r2_score
from sklearn.pipeline import Pipeline


X_train,X_test,y_train,y_test,feature_names = joblib.load('./data/prep.joblib')

lasso_m = LassoCV(alphas=np.logspace(0,4,12),cv=3,max_iter=3000, n_jobs=-1, random_state=42).fit(X_train,y_train)
y_lasso_pred = lasso_m.predict(X_test)

print(f"Lasso alpha = {lasso_m.alpha_:.2f} , R2 = {r2_score(y_test,y_lasso_pred):.3f} "
        f"RMSE = {np.sqrt(mean_squared_error(y_test,y_lasso_pred)):.0f} "
        f"MAE = {mean_absolute_error(y_test,y_lasso_pred) : .0f} "
        f"non zero feat : {int((np.abs(lasso_m.coef_) > 1e-6).sum())} ")


joblib.dump((lasso_m.coef_,feature_names),'./data/lasso_coef.joblib')

cv_rmse = np.sqrt(-cross_val_score(lasso_m, X_train, y_train, scoring='neg_mean_squared_error', cv=3)).mean()
rmse = np.sqrt(mean_squared_error(y_test,y_lasso_pred))
mae=mean_absolute_error(y_test, y_lasso_pred)
r2=r2_score(y_test, y_lasso_pred)
nonzero_lasso = int((np.abs(lasso_m.coef_) > 1e-6).sum())
alpha_lasso = getattr(lasso_m,'alpha_',np.nan)

rows.append(['Lasso (L1)', cv_rmse, rmse,mae,r2,nonzero_lasso,alpha_lasso])
fitted['Lasso (L1)'] = lasso_m



Lasso alpha = 1.00 , R2 = 0.734 RMSE = 7373 MAE =  5115 non zero feat : 162 


#Ridge alpha = 2.68 , R2 = 0.735 RMSE = 7370 MAE =  5119 non zero feat : 190

#Lasso alpha = 1.00 , R2 = 0.734 RMSE = 7373 MAE =  5115 non zero feat : 162

# We trained and evaluated Ridge(L2) and Lasso(L1) regression models to predict car prices. Both models performed similarly, with the Ridge model showing a slightly better R2 score of 0.735 and a Root Mean Squared Error (RMSE) of 7370. This indicates that the model can explain approximately 73.5% of the variance in car prices, with an average prediction error of about 7370.


In [20]:
#Interpreting the Ridge Regression Model and plotting the coeffs
coef,names = joblib.load('./data/ridge_coef.joblib')

imp_features = pd.Series(coef,index = [n.replace('num__','').replace('cat__','') for n in names]).sort_values()
print(imp_features.odometer)

print("Biggest Price pull down coefficients")
print(imp_features.head(10).round(0).to_string())

print("Biggest Price pull up coefficients ")
print(imp_features.tail(10).round(0).to_string())

print("Age and Odometer coefficients")
print(f'Age Coeff  : {imp_features.age}')
print(f'Odometer Coeff : {imp_features.odometer}')


-3992.4863134293396
Biggest Price pull down coefficients
manufacturer_fiat              -8704.0
cylinders_3 cylinders          -8333.0
manufacturer_mitsubishi        -8121.0
age                            -7972.0
manufacturer_harley-davidson   -7092.0
manufacturer_kia               -6019.0
manufacturer_hyundai           -5248.0
type_bus                       -5160.0
title_status_parts only        -5077.0
fuel_electric                  -4920.0
Biggest Price pull up coefficients 
title_status_lien                   3518.0
type_convertible                    3596.0
title_status_clean                  3654.0
manufacturer_rover                  5141.0
cylinders_10 cylinders              6114.0
manufacturer_porsche               11728.0
fuel_diesel                        12326.0
cylinders_12 cylinders             13434.0
manufacturer_tesla                 16186.0
manufacturer_infrequent_sklearn    29938.0
Age and Odometer coefficients
Age Coeff  : -7972.063977864806
Odometer Coeff : -3992.48

#Two of the numerical features namely age and odometer has the one of the biggest negative impact on the price of the car. Age has a -7972 and Odometer has -3992.

# Other major factors economy manufacturer cars like fiat, mitsubishi, kia and hyundai.

# Branded/high end cars have a positive impact on the price of the car.

# The number of cylinders for the engine also has a positive impact on the prices of the car.

# Clean title car also has a positive impact on the price.

# Electric vehicles has a negative impact on the price of the car.

In [21]:
result = pd.DataFrame(rows, columns=['Model','CV RMSE','RMSE','MAE','R2','NonZero','Alpha'])
print(result.round(3).to_string(index=False))

#plotting the Ridge coeffs

plt.rcParams.update({'figure.dpi':110,'axes.grid':True,'grid.alpha':0.3,'axes.spines.top':'False','axes.spines.right':False})
fig,ax = plt.subplots(1,2,figsize=(13,5))

ax[0].bar(result['Model'],result['R2'],color = ['#2b6cb0' if x=='Ridge (L2)' else 'g' for x in result['Model'].values])
ax[0].set_ylim(0,1)
ax[0].set_title('Test R2 by model')
ax[0].set_xlabel('Model')
ax[0].set_ylabel('R2')

print(result['R2'])

for i,v in enumerate(result['R2']):
    ax[0].text(i,v+0.02,f'{v:.3f}',ha='center')

top = pd.concat([imp_features.head(10),imp_features.tail(10)])
ax[1].barh(top.index,top.values,color=['#c53030' if x < 0 else '#2f855a' for x in top.values])
ax[1].set_title(f'Top +- Coefficients {result.iloc[0,0]}')
ax[1].set_xlabel('Effect on price')
plt.tight_layout()
plt.savefig('./images/coeff.png')
plt.show()
plt.close()

     Model  CV RMSE     RMSE      MAE    R2  NonZero  Alpha
Ridge (L2) 7367.056 7370.184 5118.559 0.735      190  2.683
Lasso (L1) 7369.127 7372.679 5114.976 0.734      162  1.000
0    0.734587
1    0.734408
Name: R2, dtype: float64


### Evaluation

With some modeling accomplished, we aim to reflect on what we identify as a high-quality model and what we are able to learn from this.  We should review our business objective and explore how well we can provide meaningful insight into drivers of used car prices.  Your goal now is to distill your findings and determine whether the earlier phases need revisitation and adjustment or if you have information of value to bring back to your client.

In [22]:
# With the R2 score the Ridge model (L2) is better. Convert the standardized coefficients into Dollars.
# Get the Dollar change in price per year and dollar change per 10k miles
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression

df=pd.read_csv('./data/df_clean.csv')
df.shape

linear = LinearRegression()

X = df[['age','odometer']]
y = df['price']

linear.fit(X,y)

print(f'The linear coeffs for age is  :  {round(linear.coef_[0],0)}')
print(f'The linear coeffs for odometer is  :  {round(linear.coef_[1]*10000,0)}')
print(f'The intercept value is :  {linear.intercept_}')




The linear coeffs for age is  :  -988.0
The linear coeffs for odometer is  :  -694.0
The intercept value is :  39423.52069540962


Using the linear regression model the coefficients for age returned is -988. It means that for every year the price of the car is reduced by 988$.

The coefficient for the Odometer is -694 for every 10k miles. This means that for every 10k miles the value of the car is reduced by 694$.



In [23]:
# Recommendations using the categorical  features
med = df['price'].median()

def premium(col):
  gr = df.groupby(col)['price'].agg(['median','count'])
  gr = gr[gr['count'] > 500]
  gr['overall'] = gr['median']-med
  return gr.sort_values('median', ascending=False)

print(premium('condition'))

print(premium('cylinders'))

print(premium('drive'))

print(premium('fuel'))

print(premium('paint_color'))

print(premium('type'))

print(premium('manufacturer'))

print(premium('title_status'))

print(premium('transmission'))



            median   count  overall
condition                          
new        22998.0     908   7003.0
good       21590.0  113558   5595.0
unknown    16900.0  136943    905.0
like new   14995.0   18725  -1000.0
excellent  11995.0   86798  -4000.0
fair        2900.0    4772 -13095.0
               median   count  overall
cylinders                             
8 cylinders   22988.0   58486   6993.0
10 cylinders  19999.0    1086   4004.0
unknown       18706.0  148989   2711.0
other         16500.0     854    505.0
6 cylinders   15999.0   82931      4.0
3 cylinders   10295.0     561  -5700.0
4 cylinders    9500.0   67461  -6495.0
5 cylinders    6500.0    1569  -9495.0
          median   count  overall
drive                            
4wd      21870.0  112850   5875.0
rwd      19999.0   47418   4004.0
unknown  15995.0  108698      0.0
fwd      10900.0   93107  -5095.0
           median   count  overall
fuel                              
diesel    33990.0   23820  17995.0
other     279

# We took median as the base as the data is right skewed. The overall column shows the value for which the typical car gets sold for. For example : The car in new condition sells for $7003 more than the median value in the data set.
#Higher the number of cylinders, higher the price of the used cars.
#Title status salvage sells for lower proce when compared to other title status.
# Tesla, Ram, Porsche and similar luxury brands sell for more price than the other economy brands like chrysler, fiat, hyundai and saturn
# Pick up trucks , coup sells at higher price than the sedans and mini vans.




### Deployment

Now that we've settled on our models and findings, it is time to deliver the information to the client.  You should organize your work as a basic report that details your primary findings.  Keep in mind that your audience is a group of used car dealers interested in fine-tuning their inventory.

# Analyzed more than 350k used car listings and used the Ridge and lasso models to fit the data. The R2 score was slightly better for Ridge which indicates a better model.
#The top 10 features that affected the price of the car both negatively and postively were determined.
# As the age of the car goes by the value of the car decreased by 988 per year.
# For every 10k miles driven the value of the car decrease by 694.
#The car in new condition sells for 7003 more than the median value in the data set followed by good, excellent and fair.
#Higher the number of cylinders, higher the price of the used cars.
#Title status salvage sells for lower proce when compared to other title status.
# Tesla, Ram, Porsche and similar luxury brands sell for more price than the other economy brands like chrysler, fiat, hyundai and saturn.
# Pick up trucks , coup sells at higher price than the sedans and mini vans.
# 4wd and rwd have higher prices than the fwd.
#White, black , orange and yello shows slightly higher median prices than the green and purple cars.

